# Agentic AI - Generate and Revise a Post on LinkedIn

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "sk-..."
os.environ["ANTHROPIC_API_KEY"] = ""

In [ ]:
if 'COLAB_GPU' in os.environ or not os.path.exists('utils') or not os.path.exists('data'):
    print("📥 Downloading required files...")
    !wget -q https://raw.githubusercontent.com/Alireza-Akhavan/Agentic_AI/refs/heads/main/utils/utils.py -P utils
    !pip install -q aisuite
    print("Setup completed")
else:
    print("Running locally - using existing files")

📥 Downloading required files...
Setup completed


In [ ]:
from dotenv import load_dotenv

load_dotenv()

import aisuite as ai

client = ai.Client()

### generate_draft Function




In [ ]:
def generate_draft(prompt: str, model: str = "openai:gpt-5.1") -> str:
    instruction = f"""
    یک پست حرفه‌ای به زبان فارسی برای لینکدین در پاسخ به موضوع زیر بنویسید.
    پست باید مختصر و جذاب باشد، حداکثر ۳۰۰ کاراکتر داشته باشد و شامل بین ۳ تا ۵ هشتگ مرتبط باشد.

    موضوع:
    {prompt}
    """
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": instruction}],
        temperature=1.0,
    )
    return response.choices[0].message.content

### reflect_on_draft Function

In [ ]:
def reflect_on_draft(draft: str, model: str = "openai:gpt-5.1") -> str:
    instruction = f"""
    پست لینکدین زیر را به دقت بررسی کنید و بازخورد سازنده‌ای ارائه دهید.
    به نکات زیر توجه کنید:
    ۱. آیا پست حداکثر ۳۰۰ کاراکتر دارد؟
    ۲. آیا بین ۳ تا ۵ هشتگ مرتبط در پست وجود دارد؟
    ۳. آیا پست جذاب، واضح و حرفه‌ای است؟
    بازخورد را در قالب یک پاراگراف ارائه دهید.

    پست:
    {draft}
    """
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": instruction}],
        temperature=1.0,
    )
    return response.choices[0].message.content

### revise_draft Function

### reflect_on_draft_with_signals Function (with quality_signals)

In [ ]:
def reflect_on_draft_with_signals(draft: str, quality_signals: dict, model: str = "openai:gpt-5.1") -> str:
    MAX_LENGTH = 300
    MIN_HASHTAGS = 3
    MAX_HASHTAGS = 5

    signal_feedback_parts = []

    if not quality_signals["within_length_limit"]:
        signal_feedback_parts.append(
            f"طول پست {quality_signals['length']} کاراکتر است در حالی که محدودیت {MAX_LENGTH} کاراکتر رعایت نشده."
        )
    if not quality_signals["within_hashtag_range"]:
        signal_feedback_parts.append(
            f"تعداد هشتگ‌ها {quality_signals['hashtag_count']} است در حالی که محدودیت بین {MIN_HASHTAGS} تا {MAX_HASHTAGS} عدد است."
        )

    signal_feedback_message = "\n".join(signal_feedback_parts)
    if signal_feedback_message:
        signal_feedback_message = "\n\nمشاهدات سیگنال‌های خام:\n" + signal_feedback_message

    instruction = f"""
    پست لینکدین زیر را به دقت بررسی کنید و بازخورد سازنده‌ای ارائه دهید.
    به نکات زیر توجه کنید:
    ۱. آیا پست حداکثر ۳۰۰ کاراکتر دارد؟
    ۲. آیا بین ۳ تا ۵ هشتگ مرتبط در پست وجود دارد؟
    ۳. آیا پست جذاب، واضح و حرفه‌ای است؟
    بازخورد را در قالب یک پاراگراف ارائه دهید.
    {signal_feedback_message}

    پست:
    {draft}
    """
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": instruction}],
        temperature=1.0,
    )
    return response.choices[0].message.content


### Comparison of `reflect_on_draft` and `reflect_on_draft_with_signals`

In [ ]:
# Create a sample post that deliberately violates limits
sample_post_violating_limits = "این یک پست کوتاه اما بدون هشتگ است که برای آزمایش بازخورد سیستم طراحی شده. هدف از این پست، بررسی تفاوت در بازخورد مدل زمانی است که سیگنال‌های خام به آن داده می‌شود در مقایسه با زمانی که فقط متن پست را می‌بیند. امیدواریم نتیجه نشان دهنده بهبود دقت و صراحت در بازخورد باشد."

print("Sample Post Violating Limits:\n" + sample_post_violating_limits + "\n")

# Analyze the raw signals for the sample post
quality_signals_for_sample = analyze_draft_signals(sample_post_violating_limits)
print("Quality Signals for Sample Post:")
for key, value in quality_signals_for_sample.items():
    print(f"  {key}: {value}")
print("\n")

# Get feedback from the original reflect_on_draft function
print("--- Feedback from Original reflect_on_draft ---")
feedback_original = reflect_on_draft(sample_post_violating_limits)
print(feedback_original)
print("\n")

# Get feedback from the new reflect_on_draft_with_signals function
print("--- Feedback from reflect_on_draft_with_signals (with raw signals) ---")
feedback_with_signals = reflect_on_draft_with_signals(sample_post_violating_limits, quality_signals_for_sample)
print(feedback_with_signals)


In [ ]:
def revise_draft(original_draft: str, reflection: str, model: str = "openai:gpt-5.1") -> str:
    instruction = f"""
    پست لینکدین زیر را با استفاده از بازخورد ارائه شده بازنویسی کنید.
    هدف بهبود وضوح، جذابیت و رعایت محدودیت‌های کاراکتر و تعداد هشتگ است.
    فقط پست اصلاح شده را برگردانید.

    پست اصلی:
    {original_draft}

    بازخورد:
    {reflection}
    """
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": instruction}],
        temperature=1.0,
    )
    return response.choices[0].message.content

### run_linkedin_workflow Function

In [ ]:
def run_linkedin_workflow(topic: str, tone: str = "حرفه‌ای", model: str = "openai:gpt-4o-mini"):
    # 1. Generate a draft
    draft = generate_draft(f"تولید یک پست لینکدین درباره {topic} با لحن {tone}", model=model)

    # 2. Reflect on the draft
    feedback = reflect_on_draft(draft, model=model)

    # 3. Revise the draft
    revised = revise_draft(draft, feedback, model=model)

    return draft, feedback, revised

### analyze_draft_signals Function

In [3]:
def analyze_draft_signals(draft: str) -> dict:
    """
    Analyzes a LinkedIn post draft for character count and hashtag count
    against predefined limits without using LLMs.

    Args:
        draft (str): The LinkedIn post draft to analyze.

    Returns:
        dict: A dictionary containing:
            - 'length': The character count of the post.
            - 'within_length_limit': True if length <= 300, False otherwise.
            - 'hashtag_count': The number of hashtags (words starting with '#').
            - 'within_hashtag_range': True if 3 <= hashtag_count <= 5, False otherwise.
    """
    MAX_LENGTH = 300
    MIN_HASHTAGS = 3
    MAX_HASHTAGS = 5

    # 1. Calculate length
    length = len(draft)
    within_length_limit = length <= MAX_LENGTH

    # 2. Calculate hashtag_count
    hashtag_count = sum(1 for word in draft.split() if word.startswith('#'))
    within_hashtag_range = MIN_HASHTAGS <= hashtag_count <= MAX_HASHTAGS

    return {
        "length": length,
        "within_length_limit": within_length_limit,
        "hashtag_count": hashtag_count,
        "within_hashtag_range": within_hashtag_range,
    }


#### Demonstration of `analyze_draft_signals`

In [4]:
# Assuming `draft_1` is available from the previous execution
if 'draft_1' in locals():
    analysis_results = analyze_draft_signals(draft_1)
    print("--- Analysis Results for Draft 1 ---")
    for key, value in analysis_results.items():
        print(f"{key}: {value}")
else:
    print("Please run the previous cells to generate `draft_1` first.")

# Example with a custom draft for testing the limits
custom_draft_too_long = "این یک پست تستی بسیار بسیار طولانی است که مطمئناً از حد ۳۰۰ کاراکتر تجاوز خواهد کرد. هدف از این تست، بررسی دقیق عملکرد تابع `analyze_draft_signals` در مواجهه با ورودی‌های بلندتر از حد مجاز است. این تابع باید بتواند به درستی طول کاراکترها را محاسبه کند و یک سیگنال `False` برای `within_length_limit` برگرداند. #هشتگ۱ #هشتگ۲"
analysis_results_long = analyze_draft_signals(custom_draft_too_long)
print("\n--- Analysis Results for Too Long Draft ---")
for key, value in analysis_results_long.items():
    print(f"{key}: {value}")

custom_draft_wrong_hashtags = "پستی با تعداد هشتگ‌های نامناسب. #کم #بیشتر_از_۵ #هشتگ_زیاد #یکی #دو #سه #چهار #پنج #شش"
analysis_results_hashtags = analyze_draft_signals(custom_draft_wrong_hashtags)
print("\n--- Analysis Results for Wrong Hashtags Draft ---")
for key, value in analysis_results_hashtags.items():
    print(f"{key}: {value}")

Please run the previous cells to generate `draft_1` first.

--- Analysis Results for Too Long Draft ---
length: 322
within_length_limit: False
hashtag_count: 2
within_hashtag_range: False

--- Analysis Results for Wrong Hashtags Draft ---
length: 86
within_length_limit: True
hashtag_count: 9
within_hashtag_range: False


### Test run_linkedin_workflow with Two Topics


#### Test Case 1: Introducing a New Training Course

In [ ]:
topic_1 = "معرفی یک دوره‌ی آموزشی جدید در حوزه هوش مصنوعی برای توسعه‌دهندگان"
draft_1, feedback_1, revised_1 = run_linkedin_workflow(topic_1)

print(f"\n--- Workflow for: {topic_1} ---")
show_output("پیش‌نویس اولیه", draft_1, background="#fff8dc", text_color="#333333")
show_output("بازخورد", feedback_1, background="#e0f7fa", text_color="#222222")
show_output("نسخه نهایی", revised_1, background="#f3e5f5", text_color="#222222")

#### Test Case 2: Announcing a Work Achievement

In [ ]:
topic_2 = "اعلام دستیابی به رکورد جدید در فروش محصولات دیجیتال در سه‌ماهه اخیر"
draft_2, feedback_2, revised_2 = run_linkedin_workflow(topic_2)

print(f"\n--- Workflow for: {topic_2} ---")
show_output("پیش‌نویس اولیه", draft_2, background="#fff8dc", text_color="#333333")
show_output("بازخورد", feedback_2, background="#e0f7fa", text_color="#222222")
show_output("نسخه نهایی", revised_2, background="#f3e5f5", text_color="#222222")